# Imports

## Importing Libraries

In [60]:
import numpy as np 
import pandas as pd 
from scipy.stats import chi2
from pandas.io.stata import StataReader

## Load and add survey weights

In [61]:
# Load the Stata dataset, extract variable labels, and drop missing survey design or outcome records.
data = pd.read_stata("MICS-child-ISRC.dta", convert_categoricals=False)
stata_file = StataReader("MICS-child-ISRC.dta")
variable_labels = stata_file.variable_labels()
survey_cols = ['hh1', 'chweight', 'strata', 'severe_wasting']
df_clean = data.dropna(subset=survey_cols).copy()
df_clean = df_clean[df_clean['chweight'] > 0].reset_index(drop=True)
variable_labels

{'hh1': 'Cluster number',
 'hh2': 'Household number',
 'ln': 'Line number',
 'ub2': 'Age of child in completed years',
 'chweight': "Children under 5's sample weight",
 'hh7a': 'District',
 'wb4': 'Age of mother/caretaker',
 'cm11': 'Total number of live births (children ever born)',
 'whz2': 'Weight for height z-score WHO',
 'strata': 'Sampling stratum (area x division)',
 'child_id': 'Unique child ID (cluster-household-line number)',
 'mother_id': 'Unique mother/caretaker ID (cluster-household-mother line number)',
 'hh_id': 'Unique household ID (cluster-household number)',
 'bd9': 'Times child ate solid/semi-solid food yesterday (7 = seven or more)',
 'diarrhoea_2w': 'Child had diarrhoea in last 2 weeks',
 'cough_2w': 'Child had cough in last 2 weeks',
 'breathing_2w': 'Child had difficulty breathing in last 2 weeks',
 'severe_wasting': 'Severe wasting (WHO WHZ < -3)',
 'mod_wasting': 'Moderate wasting (WHO -3 <= WHZ < -2)',
 'any_wasting': 'Any wasting (WHO WHZ < -2)',
 'wast_cat':

In [62]:
df_clean.shape

(22311, 49)

In [63]:
dropped_cols = ['whz2', 'mod_wasting', 'any_wasting', 'wast_cat']
try:
    df_clean = df_clean.drop(columns  = dropped_cols).copy()
except:
    print('Already Dropped!')
df_clean.shape

(22311, 45)

# Statistical Tests

## Univariate Tests

In [64]:
# Compute unweighted frequencies and survey-weighted percentages for categorical, binary, and ordinal predictors.
cat_ord_bin_vars = [
    'female', 'urban', 'division', 'water_cat', 'toilet_cat', 'fuel_cat',
    'diarrhoea_2w', 'cough_2w', 'breathing_2w', 'any_illness', 'electricity',
    'edu_level', 'wealth', 'mage_cat', 'parity_cat'
]

univariate_cat = []
total_w = df_model['chweight'].sum()

for col in cat_ord_bin_vars:
    if col in df_model.columns:
        grp = df_model.groupby(col).agg(
            unweighted_n=('chweight', 'count'),
            weighted_n=('chweight', 'sum')
        )
        grp['weighted_pct'] = (grp['weighted_n'] / total_w) * 100
        grp['variable'] = col
        grp['category'] = grp.index
        univariate_cat.append(grp.reset_index(drop=True))

df_uni_cat = pd.concat(univariate_cat, ignore_index=True)[['variable', 'category', 'unweighted_n', 'weighted_n', 'weighted_pct']]
print(df_uni_cat.to_string(index=False))

    variable  category  unweighted_n  weighted_n  weighted_pct
      female       0.0          4514 4584.669893     51.913816
      female       1.0          4256 4246.639799     48.086184
       urban       0.0          6500 6313.837986     71.493790
       urban       1.0          2270 2517.471706     28.506210
    division       1.0           733  457.692748      5.182615
    division       2.0          1916 2081.762363     23.572521
    division       3.0          1859 2224.451159     25.188236
    division       4.0          1189  917.275886     10.386635
    division       5.0           509  646.430127      7.319754
    division       6.0           904 1016.083107     11.505463
    division       7.0          1027  912.246026     10.329680
    division       8.0           633  575.368276      6.515096
   water_cat       0.0          4594 4496.127710     50.911222
   water_cat       1.0          2151 1846.958976     20.913761
   water_cat       2.0          1761 2277.724654     25

## Bivariate Tests

In [65]:
# Calculate survey-weighted mean, SD, median, and IQR for continuous predictors.
from statsmodels.stats.weightstats import DescrStatsW

cont_vars = [c for c in ['ub2', 'dds', 'fvs', 'wb4', 'cm11', 'bd9'] if c in df_model.columns]
univariate_cont = []

for col in cont_vars:
    valid = df_model[[col, 'chweight']].dropna()
    wq = DescrStatsW(valid[col], weights=valid['chweight'], ddof=0)
    q25, q50, q75 = wq.quantile([0.25, 0.50, 0.75])
    univariate_cont.append({
        'variable': col,
        'weighted_mean': wq.mean,
        'weighted_sd': wq.std,
        'weighted_median': q50,
        'weighted_iqr': q75 - q25
    })

df_uni_cont = pd.DataFrame(univariate_cont)
print(df_uni_cont.to_string(index=False))

variable  weighted_mean  weighted_sd  weighted_median  weighted_iqr
     ub2       0.522364     0.499500              1.0           1.0
     dds       2.351219     1.833790              2.0           3.0
     fvs       2.912507     2.431945              3.0           4.0
     wb4      25.853849     6.070698             25.0           9.0
    cm11       2.042436     1.083940              2.0           2.0
     bd9       4.121511     1.895864              4.0           3.0


## Multivariate Tests

In [66]:
# Generate contingency tables with unweighted counts, weighted row prevalence, and Rao-Scott adjusted chi-square tests.
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

bivariate_summary = []

for col in cat_ord_bin_vars:
    if col in df_model.columns:
        unw_tab = pd.crosstab(df_model[col], df_model['severe_wasting'])
        w_tab = pd.crosstab(df_model[col], df_model['severe_wasting'], values=df_model['chweight'], aggfunc='sum').fillna(0)
        w_row_pct = w_tab.div(w_tab.sum(axis=1), axis=0) * 100
        
        comb_tab = pd.DataFrame({
            'Unw_No_Wasting': unw_tab[0],
            'Unw_Severe_Wasting': unw_tab[1],
            'Weighted_Prev_%': w_row_pct[1].round(2)
        })
        print(f"\n--- Contingency Table: {col} ---")
        print(comb_tab)
        
        n = len(df_model)
        w_scaled = w_tab * (n / w_tab.to_numpy().sum())
        chi2_stat, _, dof, _ = stats.chi2_contingency(w_scaled, correction=False)
        
        m_full = smf.glm(f"severe_wasting ~ C({col})", data=df_model, family=sm.families.Binomial(), freq_weights=df_model['chweight']).fit(cov_type='cluster', cov_kwds={'groups': df_model['hh1']})
        r = len(df_model[col].unique()) - 1
        R_mat = np.eye(len(m_full.params))[1:]
        wald_res = m_full.wald_test(R_mat, scalar=True)
        f_stat = float(wald_res.statistic)
        p_val = float(wald_res.pvalue)
        
        bivariate_summary.append({
            'variable': col,
            'pearson_chi2': round(chi2_stat, 4),
            'rao_scott_F': round(f_stat, 4),
            'df': r,
            'p_value': round(p_val, 4)
        })

df_biv_cat = pd.DataFrame(bivariate_summary)
print("\n--- Bivariate Rao-Scott Summary ---")
print(df_biv_cat.to_string(index=False))


--- Contingency Table: female ---
        Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
female                                                     
0                 4349                 165             3.76
1                 4128                 128             2.82


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



--- Contingency Table: urban ---
       Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
urban                                                     
0                6278                 222             3.47
1                2199                  71             2.89


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



--- Contingency Table: division ---
          Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
division                                                     
1                    708                  25             3.55
2                   1844                  72             4.12
3                   1794                  65             3.26
4                   1160                  29             2.59
5                    492                  17             3.23
6                    879                  25             2.40
7                    989                  38             3.39
8                    611                  22             3.06


...

C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



--- Contingency Table: edu_level ---
           Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
edu_level                                                     
0.0                   372                  23             4.69
1.0                  1395                  51             3.78
2.0                  2148                  75             3.68
3.0                  2486                  83             2.89
4.0                  1200                  39             3.14
5.0                   876                  22             2.50

--- Contingency Table: wealth ---
        Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
wealth                                                     
1                 1817                  82             4.44
2                 1763                  66             3.61
3                 1750                  62             3.82
4                 1673                  47             2.33
5                 1474                  36             2.54


C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(



--- Contingency Table: mage_cat ---
          Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
mage_cat                                                     
1.0                 1227                  41             2.90
2.0                 2806                  89             3.08
3.0                 2250                  82             3.02
4.0                 1308                  45             3.98
5.0                  886                  36             4.27

--- Contingency Table: parity_cat ---
            Unw_No_Wasting  Unw_Severe_Wasting  Weighted_Prev_%
parity_cat                                                     
0.0                      9                   0             0.00
1.0                   3121                 104             3.18
2.0                   2956                  99             3.03
3.0                   1609                  55             3.46
4.0                    782                  35             4.56

--- Bivariate Rao-Scott Summary ---
    v

C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\statsmodels\genmod\generalized_linear_model.py:1488: SpecificationWarning: cov_type not fully supported with freq_weights
  glm_results = GLMResults(
